In [2]:
# 02_yandex_rag.ipynb

import os
import sys
import requests
import json
from pathlib import Path
from dotenv import load_dotenv

# Загружаем ключи
project_root = Path.cwd().parent
load_dotenv(project_root / '.env')

# Импорты для RAG
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions

print("✅ Библиотеки загружены")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

✅ Библиотеки загружены


In [3]:
# 1. Загружаем очищенный ПТЭ из DOCX
from docx import Document

docx_path = project_root / "data" / "для RAG птэ вариант 2.docx"
doc = Document(docx_path)
clean_text = "\n".join([para.text for para in doc.paragraphs])

print(f"📄 Загружено {len(clean_text)} символов")
print(f"📝 Первые 500 символов:\n{clean_text[:500]}")

📄 Загружено 823772 символов
📝 Первые 500 символов:
Раздел 1 пункт 1 ПТЭ. Правила технической эксплуатации железных дорог Российской Федерации (далее - Правила) устанавливают систему организации движения поездов, требования к технической эксплуатации сооружений и устройств инфраструктуры железнодорожного транспорта общего пользования (далее - инфраструктура), железнодорожных путей необщего пользования, железнодорожного подвижного состава и определяют обязанности работников железнодорожного транспорта общего и необщего пользования (далее - железно


In [4]:
# 2. Создаем чанки
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", "Раздел", "Приложение", " "]
)
chunks = splitter.split_text(clean_text)
print(f"📦 Создано {len(chunks)} чанков")

📦 Создано 1189 чанков


In [5]:
# 3. Создаем векторную базу (Chroma + эмбеддинги)
from sentence_transformers import SentenceTransformer

# Загружаем русскую модель эмбеддингов
embedding_model = SentenceTransformer('intfloat/multilingual-e5-small')

# Создаем эмбеддинги для всех чанков (может занять время)
print("🔄 Создание эмбеддингов...")
embeddings = embedding_model.encode(chunks, show_progress_bar=True)
print(f"✅ Создано {len(embeddings)} эмбеддингов размером {embeddings.shape[1]}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

/home/ruslan/RAG_PTE/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

🔄 Создание эмбеддингов...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

✅ Создано 1189 эмбеддингов размером 384


In [6]:
# 4. Функция поиска + Yandex GPT
def find_violation_yandex(remark_text, k=5):
    # Поиск похожих чанков
    remark_embedding = embedding_model.encode([remark_text])
    
    # Вычисляем косинусное расстояние
    from sklearn.metrics.pairwise import cosine_similarity
    similarities = cosine_similarity(remark_embedding, embeddings)[0]
    
    # Берем топ-k индексов
    top_indices = similarities.argsort()[-k:][::-1]
    top_chunks = [chunks[i] for i in top_indices]
    
    context = "\n\n---\n\n".join(top_chunks)
    
    # Запрос к Yandex GPT
    folder_id = os.getenv("YANDEX_FOLDER_ID")
    api_key = os.getenv("YANDEX_API_KEY")
    
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Authorization": f"Api-Key {api_key}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
Ты эксперт по ПТЭ РФ. Найди пункт правил, нарушенный в замечании.

Замечание: {remark_text}

Выдержки из документов:
{context[:3000]}

Ответь строго в формате: "В нарушение требований [точный номер пункта] [оригинальный текст замечания]"
Если точного пункта нет, напиши "НЕ НАЙДЕНО"
"""
    
    data = {
        "modelUri": f"gpt://{folder_id}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": 0.1,
            "maxTokens": 300
        },
        "messages": [{"role": "user", "text": prompt}]
    }
    
    response = requests.post(url, headers=headers, json=data, timeout=30)
    
    if response.status_code == 200:
        result = response.json()
        return result['result']['alternatives'][0]['message']['text']
    else:
        return f"ОШИБКА: {response.status_code}"

In [7]:
# 5. Тест на замечании про стрелку №47
test_remark = """на железнодорожной станции систематически не замыкается по прямому направлению 
стрелочный перевод № 47 при организации движения пассажирских поездов 
по запрещающим показаниям светофоров."""

print("🔍 Поиск нарушения...")
result = find_violation_yandex(test_remark, k=5)
print(f"\n📌 Результат:\n{result}")

🔍 Поиск нарушения...

📌 Результат:
НЕ НАЙДЕНО


In [9]:
# Диагностика с правильным импортом
from sklearn.metrics.pairwise import cosine_similarity

test_remark = """на железнодорожной станции систематически не замыкается по прямому направлению 
стрелочный перевод № 47 при организации движения пассажирских поездов 
по запрещающим показаниям светофоров."""

# Ищем чанки
remark_embedding = embedding_model.encode([test_remark])
similarities = cosine_similarity(remark_embedding, embeddings)[0]
top_indices = similarities.argsort()[-5:][::-1]

print("🔍 Топ-5 найденных чанков:\n")
for i, idx in enumerate(top_indices):
    print(f"{i+1}. Сходство: {similarities[idx]:.3f}")
    print(f"   {chunks[idx][:200]}...")
    print()

🔍 Топ-5 найденных чанков:

1. Сходство: 0.896
   станции, на которого возложено выполнение операций по приему и отправлению поездов. Проезд переключенного при срабатывании устройства контроля схода железнодорожного подвижного состава проходного свет...

2. Сходство: 0.893
   Приложение 20 пункт 39 ИДП. В случае приема поезда на железнодорожную станцию или отправления с железнодорожной станции при запрещающем показании светофора, погасших основных огнях светофора по одному...

3. Сходство: 0.891
   Раздел 1 пункт 47 ИДП. На железнодорожных станциях с наличием железнодорожных переездов, расположенных в стрелочных горловинах или на участках удаления, на которые извещение о закрытии железнодорожног...

4. Сходство: 0.888
   Приложение 14 пункт 12 ИДП. При неисправности диспетчерской централизации, когда управление одной или несколькими железнодорожными станциями невозможно, диспетчер поездной должен перевести эти железно...

5. Сходство: 0.884
   поста по указанию дежурного по железнодорож

In [12]:
# Проверяем, есть ли в чанках пункт 15
found = False
for i, chunk in enumerate(chunks):
    if "Приложение 14 пункт 15" in chunk:
        print(f"✅ Найден чанк с пунктом 15 в индексе {i}")
        print(f"   {chunk[:600]}...")
        found = True
        break

if not found:
    print("❌ Чанк с 'Приложение 14 пункт 15' не найден в базе")
    
    # Ищем в исходном тексте
    if "Приложение 14 пункт 15" in clean_text:
        print("✅ Пункт 15 есть в исходном тексте, но не попал в чанки")
    else:
        print("❌ Пункта 15 нет в исходном тексте")

✅ Найден чанк с пунктом 15 в индексе 1011
   Приложение 14 пункт 15 ИДП. Перед приемом или отправлением поезда по пригласительному сигналу или по соответствующим разрешениям при запрещающих показаниях светофоров на железнодорожных станциях, оборудованных электрической централизацией, работник, осуществляющий управление стрелками и светофорами, прежде чем воспользоваться пригласительным сигналом или выдать разрешение на прием или отправление поезда, обязан: 1) установить стрелочные рукоятки (кнопки) в положение, соответствующее положению стрелок в маршруте, и убедиться в правильности установки маршрута по индикации на аппарате управления....


In [11]:
# Функция поиска с увеличенным k и приоритетом для приложения 14
def find_violation_yandex_v2(remark_text, k=15):
    # Поиск чанков
    remark_embedding = embedding_model.encode([remark_text])
    similarities = cosine_similarity(remark_embedding, embeddings)[0]
    
    # Получаем топ-k индексов
    top_indices = similarities.argsort()[-k:][::-1]
    
    # Приоритет: добавляем бонус к чанкам с "Приложение 14"
    scored_chunks = []
    for idx in top_indices:
        score = similarities[idx]
        # Бонус +0.1 если чанк содержит "Приложение 14"
        if "Приложение 14" in chunks[idx]:
            score += 0.1
        scored_chunks.append((idx, score))
    
    # Сортируем по новому скору
    scored_chunks.sort(key=lambda x: x[1], reverse=True)
    
    # Берем топ-5 после бонуса
    top_chunks = [chunks[idx] for idx, _ in scored_chunks[:5]]
    
    context = "\n\n---\n\n".join(top_chunks)
    
    # Запрос к Yandex GPT
    folder_id = os.getenv("YANDEX_FOLDER_ID")
    api_key = os.getenv("YANDEX_API_KEY")
    
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Authorization": f"Api-Key {api_key}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
Ты эксперт по ПТЭ РФ. Найди пункт правил, нарушенный в замечании.

Замечание: {remark_text}

Выдержки из документов:
{context[:3000]}

Ответь строго в формате: "В нарушение требований [точный номер пункта] [оригинальный текст замечания]"
Если точного пункта нет, напиши "НЕ НАЙДЕНО"
"""
    
    data = {
        "modelUri": f"gpt://{folder_id}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": 0.1,
            "maxTokens": 300
        },
        "messages": [{"role": "user", "text": prompt}]
    }
    
    response = requests.post(url, headers=headers, json=data, timeout=30)
    
    if response.status_code == 200:
        result = response.json()
        return result['result']['alternatives'][0]['message']['text']
    else:
        return f"ОШИБКА: {response.status_code}"

# Тест
test_remark = """на железнодорожной станции систематически не замыкается по прямому направлению 
стрелочный перевод № 47 при организации движения пассажирских поездов 
по запрещающим показаниям светофоров."""

print("🔍 Поиск с увеличенным k и бонусом...")
result = find_violation_yandex_v2(test_remark, k=15)
print(f"\n📌 Результат:\n{result}")

🔍 Поиск с увеличенным k и бонусом...

📌 Результат:
НЕ НАЙДЕНО


In [13]:
# test_yandex_benchmark.py

import os
import sys
import requests
import json
import re
from pathlib import Path
from dotenv import load_dotenv
from tqdm import tqdm

# Загрузка ключей
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(project_root / '.env')

# Конфигурация Yandex GPT
FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")
API_KEY = os.getenv("YANDEX_API_KEY")
URL = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"

HEADERS = {
    "Authorization": f"Api-Key {API_KEY}",
    "Content-Type": "application/json"
}

def ask_yandex(prompt, max_tokens=300, temperature=0.1):
    """Запрос к Yandex GPT"""
    data = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": temperature,
            "maxTokens": max_tokens
        },
        "messages": [{"role": "user", "text": prompt}]
    }
    
    try:
        response = requests.post(URL, headers=HEADERS, json=data, timeout=30)
        if response.status_code == 200:
            result = response.json()
            return result['result']['alternatives'][0]['message']['text']
        else:
            return f"ОШИБКА: {response.status_code}"
    except Exception as e:
        return f"ИСКЛЮЧЕНИЕ: {e}"

# Список замечаний (ТОЛЬКО описание нарушения, без правильного ответа)
test_remarks = [
    "дорожный мастер при выявлении неисправности угрожающей безопасности движения не принял меры для незамедлительного остановки движения поезда",
    "руководство депо допустила машиниста к управлению локомотивом без наличия свидетельства о праве к управлению",
    "руководителями не проведена аттестация работника производственная деятельность которого связана с движением поездов",
    "составитель поездов не убедился в отсутствии препятствий для движения при проследовании технологического проезда на путь № 13",
    "дежурный по станции не организовал приготовление маршрута на весь путь следования маневрового состава",
    "дежурный по станции при возникновении отклонений в индикации аппарата управления не убедилась в своих неправильных действиях",
    "составитель поездов не наблюдал за показаниями маневровых светофоров и за положением стрелок по маршруту в момент осаживания",
    "составитель поездов допустил пропуск подвижного состава по взрезанной стрелке",
    "оператор сортировочной горки не обеспечила соединение отцепа с вагонами со скоростью не более 5 км/ч",
    "дежурный по сортировочной горке не организовал маневровую работу в соответствии с инструкцией",
    "закрепление подвижного состава было произведено тормозными башмаками с обледенелым полозом",
    "дежурный по станции при производстве маневровой работы не контролировала положение маневрового состава",
    "после допущенного взреза стрелки дежурный по станции не дала команду на немедленную остановку машинисту",
    "после допущенного взреза стрелки дежурный по станции не оформила запись в журнале осмотра",
    "машинист маневрового локомотива не наблюдал за подаваемыми сигналами светофоров и за положением стрелок при маневровой работе",
    "составитель поездов уложил тормозные башмаки без касания полозом обода колеса",
    "в локомотивном депо машинисты допускаются к управлению без служебного удостоверения и свидетельства",
    "работники локомотивных бригад не проходят предрейсовые и послерейсовые медицинские осмотры",
    "на станции смонтированные устройства САУТ не соответствуют утвержденной технической документации",
    "в путевых машинных станциях сдаются в эксплуатацию пути, не соответствующие проектной документации",
    "не обеспечено содержание путей и стрелочных переводов в исправном техническом состоянии",
    "не обеспечено ограждение контрольно-габаритными устройствами мостов и тоннеля",
    "стрелочные переводы эксплуатируются с расстоянием между сердечником и контррельсом менее 1472 мм",
    "эксплуатируются мосты с толщиной балластного слоя под шпалой менее 250 мм",
    "эксплуатируются светофоры с видимостью менее 1000 метров",
    "эксплуатируются маршрутные светофоры с видимостью менее 400 метров",
    "эксплуатируется непринятая к учету негабаритная опора контактной сети",
    "не обеспечено запирание на замки приводов секционных разъединителей",
    "эксплуатируются локомотивы, не прошедшие планово-предупредительное обслуживание",
    "эксплуатируются воздушные резервуары с истекшим сроком службы",
    "используются платформы МПД с истекшим сроком службы",
    "не определены требования к содержанию мест допуска спецподвижного состава",
    "в состав пассажирских поездов включаются локомотивы с неисправным компрессором",
    "эксплуатируются электропоезда без обслуживания устройств безопасности",
    "не установлена периодичность осмотра локомотивных устройств безопасности",
    "техобслуживание вагонов проведено без фактических проверок состояния узлов",
    "отправление вагонов с остатками груза на тормозном оборудовании",
    "к управлению спецподвижным составом допускаются машинисты без свидетельства",
    "на станции не прекращалась маневровая работа перед приемом поездов",
    "на станции не прекращалась маневровая работа перед отправлением поездов",
    "отправление поездов без коммерческого осмотра",
    "маневровая работа двумя и более локомотивами по неизолированным маршрутам",
    "не соблюдается порядок маневровой работы с выездом за границу станции",
    "не обеспечен порядок роспуска вагонов на сортировочной горке",
    "нарушения норм закрепления подвижного состава",
    "эксплуатация спецподвижного состава без тормозных башмаков",
    "локомотивы в недействующем состоянии не осматриваются комиссией",
    "отключение приборов безопасности локомотивными бригадами",
    "не ведется учет вагонов с опасными грузами в сортировочном парке",
    "систематически не замыкается стрелочный перевод при движении пассажирских поездов по запрещающим показаниям светофоров"
]

# Правильные ответы (из вашего бенчмарка)
correct_answers = {
    1: "Раздел 2 пункт 8 ПТЭ",
    2: "Раздел 2 пункт 9 ПТЭ",
    3: "Раздел 2 пункт 11 ПТЭ",
    4: "Приложение 10 пункт 24 ИДП",
    5: "Приложение 10 пункт 8 ИДП",
    6: "Приложение 14 пункт 1 ИДП",
    7: "Приложение 10 пункт 26 ИДП",
    8: "Приложение 10 пункт 26 ИДП",
    9: "Приложение 10 пункт 37 ИДП",
    10: "Приложение 10 пункт 47 ИДП",
    11: "Приложение 12 пункт 4 ИДП",
    12: "Приложение 10 пункт 18 ИДП",
    13: "Приложение 14 пункт 8 ИДП",
    14: "Приложение 14 пункт 2 ИДП",
    15: "Приложение 10 пункт 33 ИДП",
    16: "Приложение 13 пункт 3 ИДП",
    17: "Раздел 2 пункт 9 ПТЭ",
    18: "Раздел 2 пункт 10 ПТЭ",
    19: "Раздел 3 пункт 13 ПТЭ",
    20: "Раздел 3 пункт 13 ПТЭ",
    21: "Раздел 4 пункт 41 ПТЭ",
    22: "Раздел 5 пункт 49 ПТЭ",
    23: "Раздел 5 пункт 53 ПТЭ",
    24: "Раздел 5 пункт 69 ПТЭ",
    25: "Раздел 6 пункт 74 ПТЭ",
    26: "Раздел 6 пункт 76 ПТЭ",
    27: "Раздел 8 пункт 123 ПТЭ",
    28: "Раздел 8 пункт 127 ПТЭ",
    29: "Раздел 9 пункт 129 ПТЭ",
    30: "Раздел 9 пункт 130 ПТЭ",
    31: "Раздел 9 пункт 130 ПТЭ",
    32: "Раздел 9 пункт 135 ПТЭ",
    33: "Раздел 9 пункт 139 ПТЭ",
    34: "Раздел 9 пункт 141 ПТЭ",
    35: "Раздел 9 пункт 141 ПТЭ",
    36: "Раздел 9 пункт 164 ПТЭ",
    37: "Раздел 1 пункт 23 ИДП",
    38: "Раздел 1 пункт 64 ИДП",
    39: "Приложение 9 пункт 2 ИДП",
    40: "Приложение 9 пункт 3 ИДП",
    41: "Приложение 9 пункт 53 ИДП",
    42: "Приложение 10 пункт 2 ИДП",
    43: "Приложение 10 пункт 43 ИДП",
    44: "Приложение 10 пункт 50 ИДП",
    45: "Приложение 12 пункт 1 ИДП",
    46: "Приложение 12 пункт 25 ИДП",
    47: "Приложение 16 пункт 7 ИДП",
    48: "Раздел 1 пункт 68 ИДП",
    49: "Приложение 10 пункт 50 ИДП",
    50: "Приложение 14 пункт 15 ИДП"
}

def normalize(p):
    """Нормализация строки для сравнения"""
    p = p.lower().strip()
    p = re.sub(r'[№#]', '', p)
    p = re.sub(r'приложения?\s*(\d+)', r'приложение \1', p)
    p = re.sub(r'раздел\s+(\d+)', r'раздел \1', p)
    p = re.sub(r'пункт\s+(\d+)', r'пункт \1', p)
    p = re.sub(r'п\.', 'пункт', p)
    p = re.sub(r'\s+', ' ', p)
    return p.strip()

print("🚀 Запуск тестирования Yandex GPT на 50 замечаниях")
print("="*60)

results = []

for i, remark in enumerate(tqdm(test_remarks, desc="Обработка"), 1):
    prompt = f"""
Ты эксперт по ПТЭ и ИДП. Определи, какой пункт правил нарушен в ситуации.

Ситуация: {remark}

Ответь ТОЛЬКО номером пункта в формате как в примерах:
- Раздел 2 пункт 8 ПТЭ
- Приложение 14 пункт 15 ИДП
- Приложение 10 пункт 26 ИДП

Если точно не уверен, напиши "НЕ ОПРЕДЕЛЕНО"
"""
    
    predicted = ask_yandex(prompt, max_tokens=100, temperature=0.1)
    correct = correct_answers.get(i, "НЕ УКАЗАН")
    
    is_correct = normalize(predicted) == normalize(correct)
    
    results.append({
        "id": i,
        "remark": remark[:80],
        "predicted": predicted,
        "correct": correct,
        "is_correct": is_correct
    })
    
    # Небольшая задержка между запросами
    import time
    time.sleep(0.5)

# Статистика
correct_count = sum(1 for r in results if r["is_correct"])
accuracy = correct_count / 50 * 100

print("\n" + "="*60)
print(f"📊 РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ")
print("="*60)
print(f"✅ Точных совпадений: {correct_count}/50")
print(f"📈 Точность: {accuracy:.1f}%")
print("="*60)

# Сохраняем результаты
with open("yandex_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\n💾 Результаты сохранены в yandex_benchmark_results.json")

# Показываем примеры
print("\n📋 ПРИМЕРЫ ОТВЕТОВ:")
for r in results[:5]:
    status = "✅" if r["is_correct"] else "❌"
    print(f"{status} {r['id']}. Пред: {r['predicted'][:50]}... | Ожид: {r['correct']}")

🚀 Запуск тестирования Yandex GPT на 50 замечаниях


Обработка: 100%|██████████| 50/50 [00:44<00:00,  1.13it/s]


📊 РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ
✅ Точных совпадений: 1/50
📈 Точность: 2.0%

💾 Результаты сохранены в yandex_benchmark_results.json

📋 ПРИМЕРЫ ОТВЕТОВ:
✅ 1. Пред: Раздел 2 пункт 8 ПТЭ... | Ожид: Раздел 2 пункт 8 ПТЭ
❌ 2. Пред: Раздел 2 пункт 8 ПТЭ... | Ожид: Раздел 2 пункт 9 ПТЭ
❌ 3. Пред: НЕ ОПРЕДЕЛЕНО... | Ожид: Раздел 2 пункт 11 ПТЭ
❌ 4. Пред: Приложение 14 пункт 2 ИДП... | Ожид: Приложение 10 пункт 24 ИДП
❌ 5. Пред: Приложение 10 пункт 26 ИДП... | Ожид: Приложение 10 пункт 8 ИДП


Пробуем гибридный подход 

In [14]:
# 03_hybrid_search_yandex.ipynb

import os
import sys
import json
import re
import requests
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from docx import Document

# Загрузка ключей
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(project_root / '.env')

FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")
API_KEY = os.getenv("YANDEX_API_KEY")
URL = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"

HEADERS = {
    "Authorization": f"Api-Key {API_KEY}",
    "Content-Type": "application/json"
}

# 1. Загрузка документа
docx_path = project_root / "data" / "для RAG птэ вариант 2.docx"
doc = Document(docx_path)
clean_text = "\n".join([para.text for para in doc.paragraphs])

# 2. Создание чанков (уменьшенный размер для точности)
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(clean_text)
print(f"✅ Создано {len(chunks)} чанков")

# 3. Векторные эмбеддинги
embedding_model = SentenceTransformer('intfloat/multilingual-e5-small')
embeddings = embedding_model.encode(chunks, show_progress_bar=True)

# 4. BM25 индекс (поиск по ключевым словам)
tokenized_chunks = [chunk.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)

# 5. Функция гибридного поиска
def hybrid_search(query, k_vector=20, k_bm25=20, final_k=10, weight_vector=0.5, weight_bm25=0.5):
    """
    Гибридный поиск: объединяет векторный и BM25 поиск
    """
    # Векторный поиск
    query_embedding = embedding_model.encode([query])
    vector_scores = cosine_similarity(query_embedding, embeddings)[0]
    vector_indices = vector_scores.argsort()[-k_vector:][::-1]
    
    # BM25 поиск
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_indices = bm25_scores.argsort()[-k_bm25:][::-1]
    
    # Нормализация и объединение
    combined_scores = {}
    
    # Векторные оценки (нормализованные 0-1)
    for idx in vector_indices:
        combined_scores[idx] = weight_vector * vector_scores[idx]
    
    # BM25 оценки (нормализованные)
    max_bm25 = np.max(bm25_scores) if np.max(bm25_scores) > 0 else 1
    for idx in bm25_indices:
        norm_bm25 = bm25_scores[idx] / max_bm25
        combined_scores[idx] = combined_scores.get(idx, 0) + weight_bm25 * norm_bm25
    
    # Сортировка и выбор топ-k
    sorted_indices = sorted(combined_scores.keys(), key=lambda x: combined_scores[x], reverse=True)
    return sorted_indices[:final_k], [combined_scores[idx] for idx in sorted_indices[:final_k]]

# 6. Тестирование гибридного поиска на примере
test_query = "стрелочный перевод не замыкается при запрещающих показаниях светофора"
indices, scores = hybrid_search(test_query, final_k=5)

print("🔍 ГИБРИДНЫЙ ПОИСК (тест)")
print("="*60)
for i, (idx, score) in enumerate(zip(indices, scores)):
    print(f"{i+1}. Score: {score:.3f}")
    print(f"   {chunks[idx][:200]}...")
    print()

# Проверяем, попал ли пункт 15
found = False
for idx in indices:
    if "Приложение 14 пункт 15" in chunks[idx]:
        print("✅ Пункт 15 найден в результатах поиска!")
        found = True
        break

if not found:
    print("❌ Пункт 15 не попал в результаты поиска")
    # Ищем его позицию в общем списке
    for i, chunk in enumerate(chunks):
        if "Приложение 14 пункт 15" in chunk:
            print(f"   Пункт 15 находится в индексе {i}")
            break

✅ Создано 2112 чанков


/home/ruslan/RAG_PTE/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/66 [00:00<?, ?it/s]

🔍 ГИБРИДНЫЙ ПОИСК (тест)
1. Score: 0.946
   надписью "Выключено". До устранения неисправности прием и отправление поездов, а также маневровые передвижения необходимо производить при запрещающих показаниях светофоров. В маршрутах указанная стрел...

2. Score: 0.937
   акте железнодорожной станции. Движение поездов по маршрутам, в которые входят такие стрелки, должно производиться при запрещающих показаниях светофоров и опущенных вниз курбельных заслонках в электроп...

3. Score: 0.812
   сделать об этом запись в журнале осмотра и вызвать работника подразделения железнодорожной автоматики и телемеханики. До устранения неисправности работнику, осуществляющему управление стрелками и свет...

4. Score: 0.804
   путь; перевод стрелки при занятости ее подвижным составом и в случае неисправности технических средств, применяемых для контроля свободности стрелочных путевых участков; открытие светофоров, соответст...

5. Score: 0.794
   управление стрелками и светофорами, должен сделать запись в 